Nilufer Belediyesi Acik Veri Portali (CKAN) - Arsa Birim Degerleri

Bu notebook, `acikveri.nilufer.bel.tr` uzerindeki CKAN API'sini kullanarak
'Arsa Birim Degerleri' veri setinin en guncel kaynak dosyasini bulur, gecici
olarak indirir, 2010 ve sonrasi ile filtreler, Spark DataFrame'e cevirir ve
data lake'e (bronze katman) yazar. Gecici dosya, veri okunduktan sonra hemen
silinir; localde kalici veri tutulmaz.

Maliyet notu: Kaynak dosya (XLSX) 1986-2026 arasi ~180K satir icerir ve tek
parca oldugu icin indirme/parse maliyeti azalmaz, ancak 2010 oncesi satirlar
(~%46) filtrelenerek data lake'e yazilan/depolanan veri hacmi ve sonraki
Spark islemlerinin maliyeti dusurulur.

Nazik/performansli yaklasim:
- Sadece 1 API cagrisi (`package_show`) yapilir, sadece metadata icin.
- Asil veri, `/api/` altinda olmayan resmi kaynak indirme URL'inden tek
  seferde indirilir (robots.txt'teki `Disallow: /api/` kapsamina girmez).
- Paralel/tekrarli istek yoktur, `Crawl-Delay: 10` kuralina saygi gosterilir.
- Lisans: CC BY 4.0 (Nilufer Belediyesi Acik Veri Lisansi) - kullanirken atif gerekir.

In [ ]:
%pip install pandas requests openpyxl

In [ ]:
%run "./Utils"

In [ ]:
import os
import time

import pandas as pd
import requests

BASE_URL = "https://acikveri.nilufer.bel.tr"
DATASET_ID = "2026-arsa-birim-degerleri"

HEADERS = {
    "User-Agent": "aXet-Project/1.0 (acik veri arastirma amacli)",
    "Accept": "application/json",
}

In [ ]:
def get_latest_resource(dataset_id, preferred_format="XLSX"):

    url = f"{BASE_URL}/api/3/action/package_show"

    response = requests.get(
        url,
        params={"id": dataset_id},
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    resources = response.json()["result"]["resources"]

    for resource in resources:
        if resource.get("format", "").upper() == preferred_format:
            return resource

    return resources[0] if resources else None


def download_resource(resource, dest_path):

    download_url = resource["url"]

    with requests.get(download_url, headers=HEADERS, stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(dest_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 64):
                f.write(chunk)

    return dest_path

In [ ]:
resource = get_latest_resource(DATASET_ID)

print("Bulunan kaynak:", resource["name"])
print("Format:", resource["format"])
print("Son guncelleme:", resource.get("last_modified"))

temp_path = "/tmp/arsa_birim_degerleri.xlsx"

download_resource(resource, temp_path)

time.sleep(1)

print("Gecici olarak indirildi:", temp_path)

In [ ]:
df_raw = pd.read_excel(temp_path)

os.remove(temp_path)

print("Gecici dosya silindi:", temp_path)

df = df_raw[["Mahalle Adı", "Cadde Sokak Adı", "Nitelik Adı", "Yıl", "Arsa Birim Değeri"]].copy()

df.columns = ["mahalle_adi", "cadde_sokak_adi", "nitelik_adi", "yil", "arsa_birim_degeri"]

df["yil"] = pd.to_numeric(df["yil"], errors="coerce")
df["arsa_birim_degeri"] = pd.to_numeric(df["arsa_birim_degeri"], errors="coerce")

MIN_YIL = 2010

row_count_before = len(df)

df = df[df["yil"] >= MIN_YIL].reset_index(drop=True)

print(f"Filtre oncesi satir: {row_count_before}")
print(f"Filtre sonrasi satir ({MIN_YIL}+): {len(df)}")

print(df.shape)
df.head(10)

In [ ]:
df_spark = spark.createDataFrame(df)

write_to_datalake(
    df_spark,
    "abfss://axetproject@ozandatalake001.dfs.core.windows.net/axet_bronze/nilufer_arsa_birim_degerleri/"
)